# FlightAI — Asistente multimodal de vuelos con voz, herramientas y reserva simulada

**Ejercicio de la Semana 2.** Este cuaderno toma las funciones de `day5.ipynb` (chat con herramientas, `artist()` con DALL·E 3 y `talker()` con TTS) y las amplía con:

| # | Componente | Qué aporta |
|---|---|---|
| 1 | `escuchar()` — **agente nuevo de voz a texto** | Transcribe el micrófono con Whisper para poder *hablarle* al aplicativo |
| 2 | `buscar_vuelos()` — **scraping real** | Precios reales de vuelos (Avianca y demás aerolíneas) en COP |
| 3 | `reservar_vuelo()` — **herramienta nueva** | Simula la reserva y la persiste en un archivo `.json` |
| 4 | `artist()` — reusada de day5 | Imagen pop-art del destino |
| 5 | `talker()` — reusada de day5, **sin ffmpeg** | Respuesta hablada |

## Flujo del aplicativo

```
🎤 voz ──► escuchar() [Whisper] ──► texto
                                     │
                                     ▼
                            gpt-4o-mini (orquestador)
                                     │
                    ┌────────────────┼─────────────────┐
                    ▼                ▼                 ▼
            buscar_vuelos()   reservar_vuelo()  consultar_reserva()
            (scraping real)   (mock → .json)      (lee el .json)
                    │                │                 │
                    └────────────────┼─────────────────┘
                                     ▼
                       respuesta ──► talker() [TTS]  🔊
                                 └─► artist() [DALL·E 3] 🖼️
```

---

## ⚠️ Antes de ejecutar: dos advertencias honestas

**1. Avianca bloquea el scraping directo.** Se probó `https://www.avianca.com/co/es/` y responde **HTTP 403** (protección Akamai anti-bot), además de que sus *Términos y Condiciones* prohíben explícitamente la extracción automatizada. Por eso el cuaderno consulta los precios a través del agregador público de **Google Flights**, que sí devuelve tarifas reales de Avianca y del resto de aerolíneas. La celda de diagnóstico más abajo te deja comprobar el 403 por ti mismo.

**2. Esto sigue siendo scraping.** Google Flights tampoco ofrece una API pública gratuita y su HTML puede cambiar sin aviso, por lo que este código es **didáctico y frágil por diseño**. 

## 1. Dependencias

Todo lo que se necesita ya está en el entorno `llms`, salvo quizá `imageio-ffmpeg` (opcional, ver la sección 3).

In [ ]:
# Ejecuta esta celda solo si algo falta
# !pip install -q gradio openai python-dotenv requests beautifulsoup4 lxml pillow

In [ ]:
# imports
import os
import re
import json
import base64
import random
import string
import tempfile
import urllib.parse
from io import BytesIO
from pathlib import Path
from datetime import date, datetime, timedelta

import requests
from bs4 import BeautifulSoup
from PIL import Image
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

print("gradio", gr.__version__)

In [ ]:
# Inicialización

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key sin configurar")

openai = OpenAI()

MODEL        = "gpt-4o-mini"   # orquestador (texto + herramientas)
MODEL_STT    = "whisper-1"     # agente de voz a texto
MODEL_TTS    = "tts-1"         # texto a voz
MODEL_IMAGEN = "dall-e-3"      # imagen del destino

# 💰 Control de costos: cada imagen de DALL·E 3 cuesta ~4 centavos de dólar.
# Ponlo en False mientras haces pruebas y súbelo a True para la demo.
GENERAR_IMAGENES = False

# Carpeta donde se persisten las reservas simuladas
CARPETA_RESERVAS = Path("reservas")
CARPETA_RESERVAS.mkdir(exist_ok=True)
print("Reservas en:", CARPETA_RESERVAS.resolve())

## 2. ¿Se puede usar ffmpeg sin instalarlo con el gestor del sistema?

**Sí, y de hecho para este cuaderno ni siquiera hace falta.** Hay dos caminos:

### Camino A (el que usa este cuaderno): no usar ffmpeg en absoluto

`day5.ipynb` necesita ffmpeg porque reproduce el audio *dentro de Python* con `pydub.playback.play()`, y `pydub` delega la decodificación del MP3 en el binario de ffmpeg. Pero aquí la reproducción la hace **el navegador**, que decodifica MP3 de forma nativa:

* En el cuaderno → `IPython.display.Audio(ruta, autoplay=True)`
* En la app → `gr.Audio(..., autoplay=True)`, siempre que se configure `type="filepath"` y `format=None` para que Gradio entregue el archivo tal cual, sin intentar convertirlo.

Con eso `pydub` nunca entra en juego y no se necesita ffmpeg. Lo mismo aplica al micrófono: el archivo que graba el navegador se le pasa directo a Whisper, que acepta `webm`, `wav`, `mp3`, `m4a`, `ogg` y `flac`.

### Camino B: instalar ffmpeg desde pip, sin permisos de administrador

Si de todos modos quieres ffmpeg —para reusar tal cual el `talker()` de day5, o para convertir formatos—, el paquete **`imageio-ffmpeg`** trae un binario estático dentro del wheel. No requiere `sudo`, `apt` ni `brew`:

```bash
pip install imageio-ffmpeg
```

y luego le indicas a `pydub` dónde está (celda siguiente). Comprobado en este equipo: instala **ffmpeg 7.0.2 estático**.

> ⚠️ Un detalle: `imageio-ffmpeg` incluye `ffmpeg` pero **no** `ffplay`, así que `pydub.playback.play()` seguirá fallando. Sirve para *convertir*, no para *reproducir*. Otra razón más para quedarnos con el Camino A.

In [ ]:
# OPCIONAL — Camino B: ffmpeg vía pip, sin permisos de administrador.
# Este cuaderno NO lo necesita; queda aquí como referencia.

def habilitar_ffmpeg_desde_pip():
    """Instala/localiza el ffmpeg estático de imageio-ffmpeg y apunta pydub hacia él."""
    try:
        import imageio_ffmpeg
    except ImportError:
        print("Ejecuta primero:  pip install imageio-ffmpeg")
        return None

    exe = imageio_ffmpeg.get_ffmpeg_exe()

    # pydub busca ffmpeg en el PATH; se lo indicamos explícitamente
    from pydub import AudioSegment
    AudioSegment.converter = exe
    AudioSegment.ffmpeg    = exe
    AudioSegment.ffprobe   = exe

    # y lo añadimos al PATH del proceso por si otra librería lo busca
    os.environ["PATH"] = os.path.dirname(exe) + os.pathsep + os.environ.get("PATH", "")
    return exe

# Descomenta para probarlo:
# ruta = habilitar_ffmpeg_desde_pip()
# print(ruta)
# !{ruta} -version | head -1

## 3. Agente 1 — `talker()`: texto a voz *(reusado de day5, versión sin ffmpeg)*

Misma llamada a la API que en `day5.ipynb`. El único cambio es el final: en vez de `pydub` + `play()`, escribimos el MP3 en un archivo temporal y **devolvemos la ruta**. Quien reproduzca será el navegador.

In [ ]:
def talker(mensaje, voz="onyx"):
    """Convierte texto a voz y devuelve la RUTA de un .mp3 temporal.

    Voces disponibles: alloy, echo, fable, onyx, nova, shimmer.
    Devolver una ruta (en lugar de reproducir aquí) evita la dependencia de ffmpeg
    y permite que Gradio o IPython lo reproduzcan en el navegador.
    """
    if not mensaje:
        return None

    respuesta = openai.audio.speech.create(
        model=MODEL_TTS,
        voice=voz,
        input=mensaje[:4000],   # límite de la API de TTS
    )

    destino = Path(tempfile.gettempdir()) / f"flightai_{random.randint(0, 999999):06d}.mp3"
    destino.write_bytes(respuesta.content)
    return str(destino)

In [ ]:
# Prueba rápida — el navegador reproduce el mp3, sin ffmpeg de por medio
from IPython.display import Audio, display

ruta_audio = talker("Hola, soy el asistente de vuelos de FlightAI. ¿A dónde quieres viajar?")
print(ruta_audio)
display(Audio(ruta_audio, autoplay=True))

## 4. Agente 2 — `escuchar()`: voz a texto **(el agente nuevo)**

Este es el agente que faltaba en `day5.ipynb`. Recibe la ruta del archivo que graba el micrófono del navegador y lo transcribe con **Whisper**.

Detalles que importan:

* `language="es"` mejora bastante la precisión y reduce que el modelo "traduzca" solo.
* El `prompt` es un truco poco conocido: le da a Whisper vocabulario de contexto para que acierte con nombres propios y códigos IATA (`BOG`, `MDE`, `Rionegro`…).
* No se convierte el audio: Whisper acepta directamente el `webm`/`wav` del navegador. Cero ffmpeg.

In [ ]:
VOCABULARIO_VUELOS = (
    "Avianca, LATAM, Wingo, JetSMART, Copa Airlines, Bogotá, Medellín, Rionegro, Cali, "
    "Cartagena, Barranquilla, Santa Marta, Pereira, Bucaramanga, San Andrés, Miami, "
    "Madrid, Ciudad de México, Buenos Aires, Nueva York, Lima, São Paulo, "
    "BOG, MDE, CLO, CTG, BAQ, SMR, PEI, BGA, ADZ, MIA, MAD, MEX, EZE, JFK, LIM, GRU, "
    "vuelo directo, escala, ida y vuelta, económica, ejecutiva, reserva, pesos colombianos"
)

def escuchar(ruta_audio):
    """AGENTE VOZ→TEXTO. Transcribe con Whisper el audio grabado por el micrófono.

    Acepta la ruta que entrega gr.Audio(type='filepath'). No convierte formatos:
    Whisper soporta webm, wav, mp3, m4a, ogg y flac de forma nativa.
    """
    if not ruta_audio or not os.path.exists(ruta_audio):
        return ""

    try:
        with open(ruta_audio, "rb") as f:
            transcripcion = openai.audio.transcriptions.create(
                model=MODEL_STT,
                file=f,
                language="es",
                prompt=VOCABULARIO_VUELOS,   # sesga a Whisper hacia el dominio de vuelos
            )
        return (transcripcion.text or "").strip()
    except Exception as e:
        print(f"[escuchar] error de transcripción: {e}")
        return ""

print("Agente voz→texto listo.")

## 5. Agente 3 — `artist()`: imagen del destino *(reusado tal cual de day5)*

In [ ]:
def artist(city):
    """Genera una imagen pop-art del destino. Idéntica a day5.ipynb."""
    image_response = openai.images.generate(
        model=MODEL_IMAGEN,
        prompt=(f"Una imagen que representa unas vacaciones en {city}, mostrando lugares "
                f"turísticos y todo lo único de {city}, en un vibrante estilo pop-art"),
        size="1024x1024",
        n=1,
        response_format="b64_json",
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

---
## 6. Scraping de precios reales

### 6.1 Primero, la comprobación de Avianca

Ejecuta la celda siguiente para verificar de primera mano por qué no se puede raspar `avianca.com` directamente.

In [ ]:
def diagnostico_avianca(timeout=20):
    """Intenta descargar la home de Avianca y reporta qué responde el servidor.

    Sirve para documentar por qué este cuaderno NO raspa avianca.com directamente.
    """
    url = "https://www.avianca.com/co/es/"
    cabeceras = {
        "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                       "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"),
        "Accept-Language": "es-CO,es;q=0.9",
    }
    try:
        r = requests.get(url, headers=cabeceras, timeout=timeout)
        print(f"HTTP {r.status_code} — {len(r.content)} bytes")
        if r.status_code == 403:
            print("→ 403 Forbidden: Akamai Bot Manager bloquea el cliente automatizado.")
        return r.status_code
    except requests.exceptions.RequestException as e:
        print(f"→ La conexión fue rechazada o cortada: {type(e).__name__}")
        print("   (Akamai también corta el handshake TLS/HTTP2 de clientes no-navegador.)")
        return None

diagnostico_avianca()

### 6.2 La ruta que sí funciona: Google Flights

Google Flights codifica la búsqueda en un parámetro de URL llamado `tfs`: un mensaje **Protocol Buffers** serializado y luego pasado a Base64. La librería `fast-flights` hace esto mismo, pero exige `protobuf>=6`, y este entorno tiene `protobuf 5.29` porque lo fija `google-generativeai` (el que usas en `day4.ipynb`). Actualizarlo rompería Gemini.

Así que serializamos el protobuf **a mano**. Son tres funciones de cinco líneas y **cero dependencias nuevas**:

| Concepto protobuf | Implementación |
|---|---|
| Enteros *varint* (base 128, little-endian) | `_varint()` |
| Campo *length-delimited* → `(campo << 3 \| 2)` + longitud + bytes | `_campo_bytes()` |
| Campo *varint* → `(campo << 3 \| 0)` + valor | `_campo_entero()` |

El esquema del mensaje es:

```protobuf
message Aeropuerto  { string codigo = 2; }
message Tramo       { string fecha = 2; repeated Aeropuerto origen = 13;
                      repeated Aeropuerto destino = 14; }
message Busqueda    { repeated Tramo tramos = 3; repeated Passenger pasajeros = 8;
                      Seat cabina = 9; Trip tipo_viaje = 19; }
```

In [ ]:
# ---------- Serialización protobuf mínima, sin dependencias ----------

def _varint(n):
    """Codifica un entero como varint de protobuf (base 128, little-endian)."""
    salida = bytearray()
    while True:
        septeto = n & 0x7F
        n >>= 7
        salida.append(septeto | 0x80 if n else septeto)   # bit 8 = 'hay más bytes'
        if not n:
            return bytes(salida)

def _campo_bytes(numero_campo, carga):
    """Campo length-delimited (wire type 2): etiqueta + longitud + contenido."""
    return _varint(numero_campo << 3 | 2) + _varint(len(carga)) + carga

def _campo_entero(numero_campo, valor):
    """Campo varint (wire type 0): etiqueta + valor."""
    return _varint(numero_campo << 3 | 0) + _varint(valor)


# Enumeraciones del esquema de Google Flights
CABINAS   = {"economica": 1, "premium": 2, "ejecutiva": 3, "primera": 4}
IDA       = 2
IDA_VUELTA = 1

def construir_tfs(origen, destino, fecha, fecha_regreso=None, adultos=1, cabina=1):
    """Construye el parámetro `tfs` (protobuf + Base64) que espera Google Flights."""
    tipo_viaje = IDA_VUELTA if fecha_regreso else IDA

    def tramo(desde, hacia, cuando):
        return (_campo_bytes(2, cuando.encode())
                + _campo_bytes(13, _campo_bytes(2, desde.encode()))
                + _campo_bytes(14, _campo_bytes(2, hacia.encode())))

    tramos = [tramo(origen, destino, fecha)]
    if fecha_regreso:
        tramos.append(tramo(destino, origen, fecha_regreso))

    mensaje = (b"".join(_campo_bytes(3, t) for t in tramos)
               + _campo_entero(8, 1) * adultos      # un campo repetido por pasajero adulto
               + _campo_entero(9, cabina)
               + _campo_entero(19, tipo_viaje))

    return base64.b64encode(mensaje).decode()

# Comprobación
print(construir_tfs("BOG", "MDE", "2026-09-15"))

In [ ]:
# ---------- Resolución de ciudad → código IATA ----------

IATA = {
    # Colombia
    "bogota": "BOG", "bogotá": "BOG",
    "medellin": "MDE", "medellín": "MDE", "rionegro": "MDE",
    "cali": "CLO", "cartagena": "CTG", "barranquilla": "BAQ",
    "santa marta": "SMR", "pereira": "PEI", "bucaramanga": "BGA",
    "san andres": "ADZ", "san andrés": "ADZ", "armenia": "AXM",
    "cucuta": "CUC", "cúcuta": "CUC", "monteria": "MTR", "montería": "MTR",
    "villavicencio": "VVC", "leticia": "LET", "manizales": "MZL",
    "neiva": "NVA", "popayan": "PPN", "popayán": "PPN", "valledupar": "VUP",
    # América
    "miami": "MIA", "nueva york": "JFK", "new york": "JFK",
    "ciudad de mexico": "MEX", "ciudad de méxico": "MEX", "mexico": "MEX",
    "lima": "LIM", "santiago": "SCL", "buenos aires": "EZE",
    "sao paulo": "GRU", "são paulo": "GRU", "panama": "PTY", "panamá": "PTY",
    "quito": "UIO", "san jose": "SJO", "san josé": "SJO",
    "los angeles": "LAX", "orlando": "MCO", "toronto": "YYZ",
    "la habana": "HAV", "punta cana": "PUJ", "cancun": "CUN", "cancún": "CUN",
    # Europa
    "madrid": "MAD", "barcelona": "BCN", "paris": "CDG", "parís": "CDG",
    "londres": "LHR", "roma": "FCO", "lisboa": "LIS", "berlin": "BER",
    "berlín": "BER", "amsterdam": "AMS", "ámsterdam": "AMS", "munich": "MUC",
    # Otros
    "tokio": "HND", "tokyo": "HND", "estambul": "IST", "dubai": "DXB",
}

def resolver_iata(lugar):
    """Convierte 'Medellín' → 'MDE'. Si ya viene un código IATA, lo deja igual."""
    if not lugar:
        return None
    limpio = lugar.strip()
    if re.fullmatch(r"[A-Za-z]{3}", limpio):        # ya es un IATA
        return limpio.upper()
    clave = limpio.lower().strip(".,;")
    if clave in IATA:
        return IATA[clave]
    # coincidencia parcial: "aeropuerto de medellin" → MDE
    for nombre, codigo in IATA.items():
        if nombre in clave:
            return codigo
    return None

print(resolver_iata("Medellín"), resolver_iata("bog"), resolver_iata("Cartagena"), resolver_iata("Marte"))

In [ ]:
# ---------- Tarifas de referencia heredadas de day5 (plan B si el scraping falla) ----------

ticket_prices = {"londres": "$799", "parís": "$899", "tokyo": "$1400", "berlín": "$499"}

def get_ticket_price(destination_city):
    """Función original de day5.ipynb, conservada como respaldo offline."""
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

In [ ]:
# ---------- El scraper ----------

NAVEGADOR = ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
             "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36")

# Selectores CSS de Google Flights. Son clases ofuscadas: si algún día el scraper
# devuelve listas vacías, lo primero que hay que revisar es este bloque.
SEL = {
    "bloques":   'div[jsname="IWWDBc"], div[jsname="YdtKid"]',
    "items":     "ul.Rk10dc li",
    "aerolinea": ".sSHqwe.tPgKwe.ogfYpf",
    "salida":    ".wtdjmc",
    "llegada":   ".XWcVob",
    "origen":    ".G2WY5c",
    "destino":   ".c8rWCd",
    "duracion":  ".gvkrdb",
    "escalas":   ".BbR8Ec",
    "co2":       ".O7CXue",
    "precio":    ".YMlIz.FpEdX",
}

def _texto(nodo, selector):
    elemento = nodo.select_one(selector)
    return elemento.get_text(" ", strip=True) if elemento else None

def _a_pesos(texto):
    """'1.726.260 COP' → 1726260"""
    if not texto:
        return None
    encontrado = re.search(r"([\d][\d.,]*)", texto)
    if not encontrado:
        return None
    try:
        return int(encontrado.group(1).replace(".", "").replace(",", ""))
    except ValueError:
        return None

def _limpiar_aerolinea(texto):
    """'JetSMART Operado por Jetsmart Airlines S.a.s.' → 'JetSMART'"""
    if not texto:
        return "Desconocida"
    return re.split(r"\s*Operad[oa]\s+por\s*", texto)[0].strip(" ,") or "Desconocida"

def _limpiar_escalas(texto):
    """Reduce el texto verboso de escalas a algo legible para el modelo."""
    if not texto:
        return "Sin dato"
    if "irecto" in texto:
        return "Directo"
    numero = re.match(r"(\d+)\s+escalas?", texto)
    codigos = re.findall(r"\b([A-Z]{3})\b", texto)
    etiqueta = f"{numero.group(1)} escala(s)" if numero else "Con escalas"
    return f"{etiqueta} vía {', '.join(dict.fromkeys(codigos))}" if codigos else etiqueta


def buscar_vuelos(origen, destino, fecha, fecha_regreso=None, adultos=1,
                  cabina="economica", aerolinea=None, limite=6, timeout=45):
    """HERRAMIENTA 1 — Busca vuelos reales y devuelve tarifas en pesos colombianos.

    Args:
        origen, destino: ciudad ('Medellín') o código IATA ('MDE').
        fecha: fecha de ida en formato AAAA-MM-DD.
        fecha_regreso: opcional; si se envía, la búsqueda es de ida y vuelta.
        aerolinea: filtro opcional por nombre, p. ej. 'Avianca'.

    Devuelve un dict con 'vuelos' (lista ordenada por precio) y metadatos de la consulta.
    """
    iata_origen  = resolver_iata(origen)
    iata_destino = resolver_iata(destino)

    if not iata_origen or not iata_destino:
        sin_resolver = origen if not iata_origen else destino
        return {"error": f"No reconozco el aeropuerto de '{sin_resolver}'. "
                         f"Pide al usuario el código IATA de tres letras."}

    print(f"[buscar_vuelos] {iata_origen}→{iata_destino} el {fecha}"
          + (f", regreso {fecha_regreso}" if fecha_regreso else "")
          + (f", filtro aerolínea={aerolinea}" if aerolinea else ""))

    tfs = construir_tfs(iata_origen, iata_destino, fecha, fecha_regreso,
                        adultos, CABINAS.get(str(cabina).lower(), 1))
    url = ("https://www.google.com/travel/flights?tfs=" + urllib.parse.quote(tfs)
           + "&hl=es&curr=COP&gl=CO&tfu=EgQIABABIgA")

    try:
        respuesta = requests.get(
            url, timeout=timeout,
            headers={"User-Agent": NAVEGADOR, "Accept-Language": "es-CO,es;q=0.9"})
        respuesta.raise_for_status()
    except requests.exceptions.RequestException as e:
        # Plan B: la tabla estática de day5, claramente etiquetada como referencia
        referencia = get_ticket_price(str(destino))
        return {"error": f"No se pudo consultar el buscador ({type(e).__name__}).",
                "tarifa_referencia_usd": referencia,
                "fuente": "tabla estática de day5 — NO es un precio real de hoy"}

    sopa = BeautifulSoup(respuesta.text, "lxml")
    vuelos = []

    for bloque in sopa.select(SEL["bloques"]):
        for item in bloque.select(SEL["items"]):
            precio = _a_pesos(_texto(item, SEL["precio"]))
            hora_salida = _texto(item, SEL["salida"])
            if not precio or not hora_salida:
                continue                                   # fila de encabezado o publicidad

            vuelo = {
                "aerolinea":   _limpiar_aerolinea(_texto(item, SEL["aerolinea"])),
                "salida":      hora_salida,
                "llegada":     _texto(item, SEL["llegada"]),
                "origen":      _texto(item, SEL["origen"])  or iata_origen,
                "destino":     _texto(item, SEL["destino"]) or iata_destino,
                "duracion":    _texto(item, SEL["duracion"]),
                "escalas":     _limpiar_escalas(_texto(item, SEL["escalas"])),
                "huella_co2":  _texto(item, SEL["co2"]),
                "precio_cop":  precio,
            }
            if aerolinea and aerolinea.lower() not in vuelo["aerolinea"].lower():
                continue
            if vuelo not in vuelos:                        # Google repite el mejor vuelo
                vuelos.append(vuelo)

    vuelos.sort(key=lambda v: v["precio_cop"])

    if not vuelos:
        return {"error": "No se encontraron vuelos para esa ruta y fecha.",
                "sugerencia": "Prueba con otra fecha, quita el filtro de aerolínea "
                              "o verifica que la ruta exista.",
                "consulta": {"origen": iata_origen, "destino": iata_destino, "fecha": fecha}}

    return {
        "consulta": {"origen": iata_origen, "destino": iata_destino, "fecha": fecha,
                     "fecha_regreso": fecha_regreso, "adultos": adultos,
                     "cabina": cabina, "filtro_aerolinea": aerolinea},
        "moneda": "COP",
        "por_pasajero": True,
        "vuelos": vuelos[:limite],
        "fuente": "Google Flights (datos reales, consultados en vivo)",
    }

In [ ]:
# Prueba del scraper — precios reales, en pesos
prueba = buscar_vuelos("Bogotá", "Cartagena", (date.today() + timedelta(days=30)).isoformat())
print(json.dumps(prueba, indent=2, ensure_ascii=False)[:1800])

In [ ]:
# Prueba con filtro por aerolínea
prueba_avianca = buscar_vuelos("Bogotá", "Miami",
                               (date.today() + timedelta(days=45)).isoformat(),
                               aerolinea="Avianca", limite=3)
for v in prueba_avianca.get("vuelos", []):
    print(f"{v['aerolinea']:<12} {v['salida']}→{v['llegada']}  {v['duracion']:<12} "
          f"{v['escalas']:<20} $ {v['precio_cop']:,} COP".replace(",", "."))

---
## 7. La reserva simulada

Dos herramientas más. `reservar_vuelo()` genera un localizador, arma el registro y lo escribe en `reservas/<codigo>.json`; `consultar_reserva()` lo lee de vuelta.

**Sobre datos personales:** la herramienta acepta un nombre de pasajero porque una reserva sin nombre no tiene sentido, pero **no expone ningún parámetro para documento de identidad, tarjeta ni contacto**, y el registro guardado lo deja explícito. Un modelo no puede filtrar lo que nunca recibió: la forma más sólida de proteger un dato en una arquitectura de agentes es no ponerlo en el esquema de la herramienta.

In [ ]:
def _generar_localizador():
    """Código de reserva estilo aerolínea: 6 caracteres alfanuméricos."""
    alfabeto = string.ascii_uppercase + string.digits
    return "".join(random.choices(alfabeto, k=6))


def reservar_vuelo(nombre_pasajero, origen, destino, fecha, aerolinea,
                   hora_salida=None, precio_cop=None, cabina="economica",
                   fecha_regreso=None, adultos=1):
    """HERRAMIENTA 2 — Simula la reserva de un vuelo y la persiste en un .json.

    MOCKUP: no emite tiquetes ni cobra. No recibe documento de identidad,
    medio de pago ni datos de contacto, por diseño.
    """
    codigo = _generar_localizador()

    reserva = {
        "codigo_reserva": codigo,
        "estado": "SIMULADA",
        "emitida_en": datetime.now().isoformat(timespec="seconds"),
        "pasajero": {
            "nombre": nombre_pasajero,
            "cantidad_adultos": adultos,
            "documento": "NO SOLICITADO — reserva de demostración",
        },
        "itinerario": {
            "origen":        resolver_iata(origen)  or origen,
            "destino":       resolver_iata(destino) or destino,
            "fecha_ida":     fecha,
            "fecha_regreso": fecha_regreso,
            "aerolinea":     aerolinea,
            "hora_salida":   hora_salida,
            "cabina":        cabina,
        },
        "tarifa": {
            "precio_por_pasajero_cop": precio_cop,
            "total_estimado_cop": (precio_cop * adultos) if precio_cop else None,
            "moneda": "COP",
            "incluye_impuestos": "Según lo publicado por el buscador; verificar con la aerolínea",
        },
        "aviso": ("Reserva SIMULADA con fines educativos. No constituye un contrato de "
                  "transporte ni una emisión real. Ningún dato personal real debe "
                  "registrarse aquí."),
    }

    ruta = CARPETA_RESERVAS / f"{codigo}.json"
    ruta.write_text(json.dumps(reserva, indent=2, ensure_ascii=False), encoding="utf-8")

    print(f"[reservar_vuelo] reserva {codigo} guardada en {ruta}")
    reserva["archivo"] = str(ruta)
    return reserva


def consultar_reserva(codigo_reserva):
    """HERRAMIENTA 3 — Recupera una reserva simulada desde su archivo .json."""
    ruta = CARPETA_RESERVAS / f"{str(codigo_reserva).strip().upper()}.json"
    print(f"[consultar_reserva] buscando {ruta}")
    if not ruta.exists():
        return {"error": f"No existe ninguna reserva con el código {codigo_reserva}."}
    return json.loads(ruta.read_text(encoding="utf-8"))

In [ ]:
# Prueba del mockup de reserva
demo = reservar_vuelo("Pasajero de Prueba", "Bogotá", "Cartagena",
                      (date.today() + timedelta(days=30)).isoformat(),
                      aerolinea="Avianca", hora_salida="07:35", precio_cop=289900)
print(json.dumps(consultar_reserva(demo["codigo_reserva"]), indent=2, ensure_ascii=False))

---
## 8. Declaración de las herramientas

`day5.ipynb` declaraba una sola herramienta y la resolvía con un `handle_tool_call` que leía `message.tool_calls[0]`. Eso se queda corto aquí por dos razones: ahora hay tres herramientas, y el modelo necesita **encadenarlas** (buscar → reservar → confirmar). El orquestador de la sección 9 corrige ambas cosas.

In [ ]:
herramienta_buscar = {
    "name": "buscar_vuelos",
    "description": (
        "Busca vuelos reales con precios actuales en pesos colombianos. Úsala siempre que "
        "el cliente pregunte por disponibilidad, horarios o precios de un vuelo. Devuelve "
        "una lista ordenada de menor a mayor precio, por pasajero."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "origen":   {"type": "string", "description": "Ciudad o código IATA de origen, p. ej. 'Bogotá' o 'BOG'"},
            "destino":  {"type": "string", "description": "Ciudad o código IATA de destino, p. ej. 'Cartagena' o 'CTG'"},
            "fecha":    {"type": "string", "description": "Fecha de ida en formato AAAA-MM-DD"},
            "fecha_regreso": {"type": "string", "description": "Fecha de regreso AAAA-MM-DD. Omítela si es solo ida."},
            "adultos":  {"type": "integer", "description": "Número de pasajeros adultos. Por defecto 1."},
            "cabina":   {"type": "string", "enum": ["economica", "premium", "ejecutiva", "primera"],
                         "description": "Clase de cabina. Por defecto 'economica'."},
            "aerolinea": {"type": "string", "description": "Filtro opcional por aerolínea, p. ej. 'Avianca'."},
        },
        "required": ["origen", "destino", "fecha"],
        "additionalProperties": False,
    },
}

herramienta_reservar = {
    "name": "reservar_vuelo",
    "description": (
        "Crea una reserva SIMULADA de un vuelo y la guarda en un archivo .json. Llámala solo "
        "después de que el cliente haya elegido explícitamente un vuelo concreto de los que "
        "le mostraste y te haya dado su nombre. Nunca pidas documento de identidad ni datos "
        "de pago."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "nombre_pasajero": {"type": "string", "description": "Nombre del pasajero tal como lo dio el cliente"},
            "origen":      {"type": "string", "description": "Ciudad o IATA de origen"},
            "destino":     {"type": "string", "description": "Ciudad o IATA de destino"},
            "fecha":       {"type": "string", "description": "Fecha de ida AAAA-MM-DD"},
            "aerolinea":   {"type": "string", "description": "Aerolínea del vuelo elegido"},
            "hora_salida": {"type": "string", "description": "Hora de salida del vuelo elegido"},
            "precio_cop":  {"type": "integer", "description": "Precio por pasajero en COP, tomado de buscar_vuelos"},
            "cabina":      {"type": "string", "description": "Clase de cabina"},
            "fecha_regreso": {"type": "string", "description": "Fecha de regreso AAAA-MM-DD, si aplica"},
            "adultos":     {"type": "integer", "description": "Número de pasajeros adultos"},
        },
        "required": ["nombre_pasajero", "origen", "destino", "fecha", "aerolinea"],
        "additionalProperties": False,
    },
}

herramienta_consultar = {
    "name": "consultar_reserva",
    "description": ("Recupera una reserva ya creada a partir de su código de seis caracteres. "
                    "Úsala cuando el cliente pregunte por el estado o los detalles de su reserva."),
    "parameters": {
        "type": "object",
        "properties": {
            "codigo_reserva": {"type": "string", "description": "Localizador de seis caracteres, p. ej. 'K7M2QX'"},
        },
        "required": ["codigo_reserva"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": h}
         for h in (herramienta_buscar, herramienta_reservar, herramienta_consultar)]

# Mapa nombre declarado → función de Python
FUNCIONES = {
    "buscar_vuelos":     buscar_vuelos,
    "reservar_vuelo":    reservar_vuelo,
    "consultar_reserva": consultar_reserva,
}

print(f"{len(tools)} herramientas declaradas:", ", ".join(FUNCIONES))

## 9. El orquestador

Tres mejoras sobre el `chat()` de day5:

1. **Bucle, no un solo turno.** El modelo puede pedir herramientas varias veces seguidas (buscar → reservar → confirmar). Se itera hasta que responda con texto, con un tope para no gastar tokens de más si se queda en ciclo.
2. **Todas las `tool_calls`, no solo la primera.** El modelo puede pedir dos búsquedas en paralelo, y la API exige una respuesta `role="tool"` por cada `tool_call_id`. Omitir una rompe la conversación.
3. **Los errores viajan de vuelta al modelo.** Si el scraping falla o los argumentos son inválidos, se devuelve el error como resultado de la herramienta en vez de lanzar una excepción. El modelo puede entonces disculparse o reintentar con otros datos, en vez de tumbar la interfaz.

In [ ]:
def ejecutar_tools(mensaje_modelo, estado):
    """Ejecuta TODAS las tool_calls de un turno y devuelve una respuesta por cada una.

    `estado` es un dict mutable donde se anotan efectos para la interfaz
    (destino para la imagen, última reserva creada).
    """
    respuestas = []

    for llamada in mensaje_modelo.tool_calls:
        nombre = llamada.function.name
        try:
            argumentos = json.loads(llamada.function.arguments or "{}")
        except json.JSONDecodeError:
            argumentos = {}

        funcion = FUNCIONES.get(nombre)
        if funcion is None:
            resultado = {"error": f"Herramienta desconocida: {nombre}"}
        else:
            try:
                resultado = funcion(**argumentos)
            except TypeError as e:
                resultado = {"error": f"Argumentos inválidos para {nombre}: {e}"}
            except Exception as e:
                # El modelo recibe el error y decide qué decirle al cliente
                resultado = {"error": f"{type(e).__name__}: {e}"}

        # Efectos laterales para la UI
        if nombre == "buscar_vuelos" and isinstance(resultado, dict) and resultado.get("vuelos"):
            estado["ciudad_destino"] = argumentos.get("destino")
        if nombre in ("reservar_vuelo", "consultar_reserva") and isinstance(resultado, dict) \
                and resultado.get("codigo_reserva"):
            estado["reserva"] = resultado

        respuestas.append({
            "role": "tool",
            "tool_call_id": llamada.id,
            "content": json.dumps(resultado, ensure_ascii=False),
        })

    return respuestas

In [ ]:
system_message = f"""Eres el asistente de vuelos de FlightAI. Hablas español colombiano, de forma
cordial y directa. Hoy es {date.today().isoformat()}.

Tu trabajo:
1. Ayudar al cliente a encontrar vuelos usando la herramienta buscar_vuelos.
2. Presentar máximo 3 opciones, siempre con aerolínea, horario, escalas y precio en pesos
   colombianos formateado con separador de miles (ej: $ 289.900 COP).
3. Cuando el cliente elija una opción concreta y te dé su nombre, crear la reserva con
   reservar_vuelo y confirmarle el código de seis caracteres.

Reglas importantes:
- Las fechas siempre en formato AAAA-MM-DD al llamar herramientas. Si el cliente dice
  "el próximo viernes" o "en dos semanas", calcula la fecha a partir de la de hoy.
- Si te falta el origen, el destino o la fecha, pregunta por lo que falte antes de buscar.
- Nunca inventes precios, horarios ni disponibilidad: si no llamaste a la herramienta, no
  tienes el dato. Si la herramienta devuelve un error, dilo con honestidad.
- Los precios son por pasajero y pueden cambiar; recuérdalo al confirmar.
- NUNCA pidas cédula, pasaporte, datos de tarjeta, teléfono ni correo. La reserva es una
  simulación y no los necesita. Si el cliente los ofrece, dile amablemente que no los
  registres.
- Aclara siempre que la reserva es SIMULADA y no constituye una emisión real de tiquetes.

Respuestas breves: es una conversación hablada, así que evita listas largas y textos que
suenen mal leídos en voz alta."""

print(system_message[:300], "...")

In [ ]:
# ---------- Compatibilidad del historial entre Gradio y OpenAI ----------

def normalizar_historial(history):
    """Convierte el historial que entrega Gradio en mensajes válidos para OpenAI.

    Gradio 6 devuelve cada mensaje con campos extra (`metadata`, `options`) y con el
    contenido troceado en partes: [{"type": "text", "text": "hola"}]. La API de chat
    rechaza esos campos adicionales, así que reducimos todo a {'role', 'content'} con
    el texto plano. En Gradio 4/5 los mensajes ya vienen limpios y la función los deja
    intactos, de modo que el cuaderno funciona en las tres versiones.
    """
    limpio = []

    for mensaje in history or []:
        if isinstance(mensaje, dict):
            rol, contenido = mensaje.get("role"), mensaje.get("content")
        else:                                   # objeto gr.ChatMessage
            rol, contenido = getattr(mensaje, "role", None), getattr(mensaje, "content", None)

        if isinstance(contenido, list):         # contenido troceado en partes
            partes = [p.get("text", "") if isinstance(p, dict) else str(p) for p in contenido]
            contenido = " ".join(p for p in partes if p)

        if rol in ("user", "assistant", "system") and isinstance(contenido, str) and contenido.strip():
            limpio.append({"role": rol, "content": contenido})

    return limpio


# Comprobación con el formato "gordo" de Gradio 6
_prueba = [{"role": "user", "metadata": None, "options": None,
            "content": [{"text": "hola", "type": "text"}]}]
print(normalizar_historial(_prueba))

In [ ]:
MAX_ITERACIONES = 5   # tope de rondas de herramientas por turno

def chat(history):
    """Orquestador. Recibe el historial de Gradio y devuelve texto, imagen, audio y reserva."""
    historial = normalizar_historial(history)   # ← imprescindible en Gradio 6
    mensajes = [{"role": "system", "content": system_message}] + historial
    estado = {}
    respuesta_texto = "Lo siento, no pude completar la solicitud."

    for _ in range(MAX_ITERACIONES):
        respuesta = openai.chat.completions.create(
            model=MODEL, messages=mensajes, tools=tools)
        mensaje = respuesta.choices[0].message

        if mensaje.tool_calls:
            mensajes.append(mensaje)
            mensajes.extend(ejecutar_tools(mensaje, estado))
            continue          # otra vuelta: el modelo ya tiene los resultados

        respuesta_texto = mensaje.content or respuesta_texto
        break
    else:
        respuesta_texto = ("Estoy dando demasiadas vueltas con esta consulta. "
                           "¿Puedes darme el origen, el destino y la fecha otra vez?")

    history = historial + [{"role": "assistant", "content": respuesta_texto}]

    # Agentes multimodales de salida
    imagen = None
    if GENERAR_IMAGENES and estado.get("ciudad_destino"):
        try:
            imagen = artist(estado["ciudad_destino"])
        except Exception as e:
            print(f"[artist] no se pudo generar la imagen: {e}")

    audio = None
    if VOZ_ACTIVADA:
        try:
            audio = talker(respuesta_texto)
        except Exception as e:
            print(f"[talker] no se pudo generar el audio: {e}")

    return history, imagen, audio, estado.get("reserva")


VOZ_ACTIVADA = True   # ponlo en False si no quieres que hable

---
## 10. La interfaz de Gradio

Dos entradas: el cuadro de texto de siempre y **el micrófono**, que dispara el agente de voz a texto.

Tres detalles de configuración que importan:

* `gr.Audio(sources=["microphone"], type="filepath", format=None)` — con `format=None`, Gradio entrega el archivo del navegador **tal cual**, sin convertirlo. Si se deja `format="wav"`, Gradio intentaría convertir con `pydub` y ahí sí exigiría ffmpeg. Este es exactamente el punto donde el cuaderno se libra de la dependencia.
* `gr.Chatbot` en **Gradio 6 ya no acepta `type="messages"`** (el formato de mensajes es ahora el único). El código de `day5.ipynb` fallaría con `TypeError` en esta versión, así que abajo hay un pequeño *shim* que funciona en Gradio 4, 5 y 6.
* **El historial que entrega Gradio 6 no es válido para OpenAI.** Cada mensaje llega con campos extra (`metadata`, `options`) y con el contenido troceado en partes (`[{"type": "text", "text": "hola"}]`). La API de chat rechaza esos campos, así que `normalizar_historial()` lo limpia antes de cada llamada. Es el error que más cuesta diagnosticar al portar el código de `day5.ipynb` a Gradio 6, porque no falla al construir la interfaz sino al enviar el primer mensaje.

In [ ]:
def crear_chatbot(**kwargs):
    """Shim de compatibilidad: Gradio 4/5 exige type='messages'; Gradio 6 ya no lo acepta."""
    try:
        return gr.Chatbot(type="messages", **kwargs)
    except TypeError:
        return gr.Chatbot(**kwargs)


def enviar_texto(mensaje, history):
    """Añade el mensaje escrito al historial y limpia la caja de texto."""
    history = normalizar_historial(history)
    if not mensaje or not mensaje.strip():
        return "", history
    return "", history + [{"role": "user", "content": mensaje.strip()}]


def enviar_voz(ruta_audio, history):
    """AGENTE VOZ→TEXTO en acción: transcribe el micrófono y lo mete al historial."""
    history = normalizar_historial(history)
    texto = escuchar(ruta_audio)
    if not texto:
        return history, "No se entendió el audio. Intenta de nuevo."
    return history + [{"role": "user", "content": texto}], f"🎤 Escuché: “{texto}”"


with gr.Blocks(title="FlightAI — asistente de vuelos por voz") as ui:
    gr.Markdown("# ✈️ FlightAI\n"
                "Háblale o escríbele. Busca **vuelos reales** y simula tu reserva en un `.json`.")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = crear_chatbot(height=460, label="Conversación")
            transcripcion = gr.Markdown("")
            with gr.Row():
                entrada = gr.Textbox(
                    label="Escribe tu mensaje",
                    placeholder="Quiero volar de Bogotá a Cartagena el 15 de septiembre",
                    scale=4)
                microfono = gr.Audio(
                    sources=["microphone"],
                    type="filepath",
                    format=None,          # ← clave: sin conversión, sin ffmpeg
                    label="🎤 O háblame",
                    scale=2)
            limpiar = gr.Button("Limpiar conversación")

        with gr.Column(scale=2):
            imagen_destino = gr.Image(height=320, label="Tu destino")
            audio_salida = gr.Audio(label="🔊 Respuesta hablada", autoplay=True)
            reserva_json = gr.JSON(label="Reserva simulada (.json)")

    salidas = [chatbot, imagen_destino, audio_salida, reserva_json]

    # Ruta 1: entrada escrita
    entrada.submit(enviar_texto, [entrada, chatbot], [entrada, chatbot]).then(
        chat, chatbot, salidas)

    # Ruta 2: entrada hablada → escuchar() → mismo orquestador
    microfono.stop_recording(enviar_voz, [microfono, chatbot], [chatbot, transcripcion]).then(
        chat, chatbot, salidas).then(
        lambda: None, None, microfono)   # libera el micrófono para la siguiente grabación

    limpiar.click(lambda: ([], None, None, None, ""),
                  None, salidas + [transcripcion], queue=False)

ui.launch(inbrowser=True)